# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** Ключникова Дарья Алексеевна
**Группа:** ИНФА-221
**Вариант:** 7 (предметная область: Нефтегазовая отрасль: добыча и переработка)

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

Подключение стандартных и внешних библиотек, определение XML-парсера, выбор реализации расстояния Левенштейна и определение общих параметров среды.

In [1]:
!pip install -q Levenshtein pandas lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.2 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import csv
import os
import re
import unicodedata
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import pandas as pd

# --- XML backend -----------------------------------------------------------
try:
    from lxml import etree as ET

    LXML_AVAILABLE = True
except ImportError:  # pragma: no cover - fallback для «голого» окружения
    import xml.etree.ElementTree as ET

    LXML_AVAILABLE = False

# --- Levenshtein backend ---------------------------------------------------
try:
    import Levenshtein as _lev

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (C-реализация)."""
        return _lev.distance(a, b)

    LEV_BACKEND = "Levenshtein (C)"
except ImportError:  # pragma: no cover

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (чистый Python, O(n*m) время, O(m) память)."""
        if a == b:
            return 0
        if not a:
            return len(b)
        if not b:
            return len(a)
        previous: List[int] = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current: List[int] = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(
                    min(
                        previous[j] + 1,          # удаление
                        current[j - 1] + 1,       # вставка
                        previous[j - 1] + (char_a != char_b),  # замена
                    )
                )
            previous = current
        return previous[-1]

    LEV_BACKEND = "pure-python fallback"

XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"
WORK_DIR = os.getcwd()


def indent_tree(element, level: int = 0, space: str = "  ") -> None:
    """Рекурсивно расставить отступы в XML-дереве (работает и с lxml, и с ET)."""
    pad = "\n" + space * level
    if len(element):
        if not (element.text or "").strip():
            element.text = pad + space
        for child in element:
            indent_tree(child, level + 1, space)
        if not (element.tail or "").strip():
            element.tail = pad
        if not (element[-1].tail or "").strip():
            element[-1].tail = pad
    elif level and not (element.tail or "").strip():
        element.tail = pad

pd.set_option("display.max_colwidth", 46)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

print(f"XML backend        : {'lxml' if LXML_AVAILABLE else 'xml.etree.ElementTree'}")
print(f"Levenshtein backend: {LEV_BACKEND}")
print(f"pandas             : {pd.__version__}")
print(f"Рабочий каталог    : {WORK_DIR}")

XML backend        : lxml
Levenshtein backend: Levenshtein (C)
pandas             : 2.2.3
Рабочий каталог    : /content


## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Создание **демонстрационных** файлов `demo.tbx` и `demo.tmx` для автотестов из Блока 5.

Создание словаря и памяти переводов по своей области вручную для выполнения лабораторной работы.

In [6]:
DEMO_TBX = """<?xml version="1.0" encoding="UTF-8"?>
<martif type="TBX" xml:lang="en">
  <martifHeader><fileDesc><titleStmt><title>Demo glossary</title></titleStmt>
  </fileDesc></martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en">
        <tig><term>flight controller</term>
        <termNote type="partOfSpeech">noun</termNote>
        <descrip type="definition">Onboard flight computer.</descrip></tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp><term>полётный контроллер</term></termGrp></ntig>
      </langSet>
    </termEntry>
    <termEntry id="tid-002">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>payload</term>
        <termNote type="partOfSpeech">noun</termNote></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>целевая нагрузка</term>
        </termGrp></ntig></langSet>
    </termEntry>
    <termEntry id="tid-003">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>telemetry link</term></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>канал телеметрии</term>
        </termGrp></ntig></langSet>
    </termEntry>
  </body></text>
</martif>
"""

DEMO_TMX = """<?xml version="1.0" encoding="UTF-8"?>
<tmx version="1.4">
  <header creationtool="demo" creationtoolversion="1.0" segtype="sentence"
          o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The flight controller stabilises the aircraft.</seg></tuv>
      <tuv xml:lang="ru"><seg>Полётный контроллер стабилизирует аппарат.</seg></tuv>
    </tu>
    <tu tuid="2">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The operator checks the telemetry link.</seg></tuv>
      <tuv xml:lang="ru"><seg>Оператор проверяет канал телеметрии.</seg></tuv>
    </tu>
    <tu tuid="3">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The payload is mounted under the fuselage.</seg></tuv>
      <tuv xml:lang="ru"><seg>Целевая нагрузка устанавливается под фюзеляжем.</seg></tuv>
    </tu>
  </body>
</tmx>
"""

with open("demo.tbx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TBX)
with open("demo.tmx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TMX)

print("Созданы файлы:", [f for f in os.listdir(".") if f.endswith((".tbx", ".tmx"))])

Созданы файлы: ['demo.tmx', 'demo.tbx', 'variant_7.tbx', 'variant_7.tmx']


In [4]:
TERMS: List[Dict[str, str]] = [
    {"en": "amplitude", "ru": "амплитуда", "pos": "noun",
     "def_en": "Difference between the maximum displacement of a wave and the point of no displacement or the null point."},
    {"en": "aquifer", "ru": "водоносный горизонт", "pos": "noun",
     "def_en": "Any water-bearing formation encountered while drilling."},
    {"en": "appraisal", "ru": "оценка месторождения", "pos": "noun",
     "def_en": "The phase of petroleum operations that immediately follows successful exploratory drilling."},
    {"en": "air drilling", "ru": "пневматическое бурение", "pos": "noun",
     "def_en": "Air drilling is a drilling technique whereby gases (typically compressed air or nitrogen) are used to cool the drill bit and lift cuttings out of the wellbore instead of the more conventional use of liquids."},
    {"en": "asphaltenes", "ru": "асфальтены", "pos": "noun",
     "def_en": "Asphaltenes are organic materials that contain aromatic and naphthenic ring compounds containing nitrogen, sulfur, and oxygen molecules."},
    {"en": "asphalt", "ru": "асфальт", "pos": "noun",
     "def_en": "A solid or nearly solid form of bitumen that can melt upon heating and contains impurities such as nitrogen, oxygen, and sulfur."},
    {"en": "bit", "ru": "буровое долото", "pos": "noun",
     "def_en": "The tool used to crush or cut rock."},
    {"en": "blowout", "ru": "выброс", "pos": "noun",
     "def_en": "An uncontrolled flow of reservoir fluids into the wellbore and sometimes catastrophically to the surface."},
    {"en": "blowout preventer (BOP)", "ru": "противовыбросовый превентор (ПВО)", "pos": "noun",
     "def_en": "Equipment installed on the wellhead to prevent the escape of fluids under pressure from the wellbore during drilling, completion, or workover operations."},
    {"en": "borehole", "ru": "ствол скважины", "pos": "noun",
     "def_en": "The wellbore itself, including the open hole or uncased portion of the well."},
#10
    {"en": "beam pump", "ru": "глубинный насос штангового типа", "pos": "noun",
     "def_en": "An artificial lift pumping system using a surface power source to drive a downhole pump assembly."},
    {"en": "ball valve", "ru": "шаровой клапан", "pos": "noun",
     "def_en": "A valve that uses a spherical closure element (ball) that is rotated through 90° to open and close the valve."},
    {"en": "barrel (bbl)", "ru": "баррель", "pos": "noun",
     "def_en": "A measure of volume for petroleum products."},
    {"en": "burner", "ru": "горелка", "pos": "noun",
     "def_en": "A crucial piece of every oil or gas heating system, designed to heat air or water."},
    {"en": "backpressure", "ru": "противодавление", "pos": "noun",
     "def_en": "Pressure registered on equipment or devices when fluid flows through."},
    {"en": "blinding", "ru": "установка заглушек", "pos": "noun",
     "def_en": "A process that refers to the complete sealing off of any other refinery equipment by use of a metal plate that is appropriately designed so that it is crammed onto the end of a flanged vessel or pipe or even inserted between two flanges."},
    {"en": "casing head", "ru": "обсадная головка", "pos": "noun",
     "def_en": "The adapter between the first casing string and either the BOP stack (during drilling) or the wellhead (after completion)."},
    {"en": "christmas tree", "ru": "фонтанное устьевое оборудование", "pos": "noun",
     "def_en": "The set of valves, spools, and fittings connected to the top of a well to direct and control the flow of formation fluids from the well."},
    {"en": "compressibility", "ru": "сжимаемость", "pos": "noun",
     "def_en": "The relative change in fluid volume related to a unit change in pressure."},
    {"en": "cementing", "ru": "цементирование", "pos": "noun",
     "def_en": "Permanently securing the casing to the wellbore or plugging and sealing the wellbore with cement."},
#20
    {"en": "choke", "ru": "дроссельный вентиль", "pos": "noun",
     "def_en": "A device incorporating an orifice that is used to control fluid flow rate or downstream system pressure."},
    {"en": "condensate", "ru": "конденсат", "pos": "noun",
     "def_en": "A natural gas liquid with a lower vapor pressure than natural gasoline and liquefied petroleum gas."},
    {"en": "condensate gas", "ru": "газовый конденсат", "pos": "noun",
     "def_en": "Hydrocarbon liquid dissolved in saturated natural gas that comes out of the solution when the pressure drops below the dewpoint."},
    {"en": "corrosion inhibitor", "ru": "ингибитор коррозии", "pos": "noun",
     "def_en": "A chemical additive used in acid treatments to protect iron and steel components in the wellbore and treating equipment from the corrosive fluid."},
    {"en": "corrosion-resistant alloy", "ru": "коррозионностойкий сплав", "pos": "noun",
     "def_en": "A specially formulated material used for completion components in wells that is likely to present corrosion problems."},
    {"en": "density", "ru": "плотность", "pos": "noun",
     "def_en": "Degree of consistency measured by the quantity of mass per unit volume."},
    {"en": "dehydrate", "ru": "обезвоживать", "pos": "verb",
     "def_en": "To remove water from a substance."},
    {"en": "demulsification", "ru": "деэмульгация", "pos": "noun",
     "def_en": "A mixture of chemicals breaks the emulsion by destroying or weakening the stabilizing film around the dispersed drops."},
    {"en": "dense phase", "ru": "плотная фаза", "pos": "noun",
     "def_en": "A state when fluid exists above both the cricondenbar pressure and the critical temperaturen."},
    {"en": "derrick floor", "ru": "рабочая площадка буровой установки", "pos": "noun",
     "def_en": "The relatively small work area in which the rig crew conducts operations, usually adding or removing drillpipe to or from the drillstring."},
#30
    {"en": "desalter", "ru": "обессоливающая установка", "pos": "noun",
     "def_en": "A device used in petroleum refineries to remove inorganic salts, water, and sediment from the incoming petroleum crude oil feedstock before it is refined."},
    {"en": "decompression", "ru": "декомпрессия", "pos": "noun",
     "def_en": "A reduction in pressure; release from pressure or compression."},
    {"en": "exploration", "ru": "геологоразведочные работы", "pos": "noun",
     "def_en": "The act of searching for the discovery of information or resources. Oil and gas exploration encompasses the processes and methods of locating potential oil and gas drilling and extraction sites."},
    {"en": "hydraulic fracturing", "ru": "гидравлический разрыв пласта", "pos": "noun",
     "def_en": "The practice of pumping special fluids down the wellbore under high pressure."},
    {"en": "marginal well", "ru": "низкорентабельная скважина", "pos": "noun",
     "def_en": "A well that, for reasons of depletion or naturally low productivity, is nearing the limits of viable production and profitability."},
    {"en": "sedimentary basin", "ru": "осадочный бассейн", "pos": "noun",
     "def_en": "A depression in the crust of the Earth caused by plate tectonic activity and subsidence, in which sediments accumulate."},
    {"en": "sedimentary rocks", "ru": "осадочные породы", "pos": "noun",
     "def_en": "Rocks generated as a result of organic and inorganic matter settling on bottoms of water basins and surface of continents."},
    {"en": "pressure", "ru": "давление", "pos": "noun",
     "def_en": "Force distributed over a surface, usually measured in pounds force per square inch."},
    {"en": "positive displacement motor", "ru": "гидравлический забойный двигатель","pos": "noun",
     "def_en": "A downhole motor used in the oil field to drive the drill bit or other downhole tools during directional drilling or performance drilling applications."},
    {"en": "recover", "ru": "восполнять; возвращать", "pos": "verb",
     "def_en": "Getting back, regaining, or making up for."},
#40
    {"en": "rotary drilling", "ru": "роторное бурение", "pos": "noun",
     "def_en": "A method of making a hole that relies on a continuous circular motion of the bit to break rock at the bottom of the hole."},
    {"en": "sweetening", "ru": "десульфурация", "pos": "noun",
     "def_en": "A process used to remove hydrogen sulfide [H2S] and carbon dioxide [CO2] from a gas stream."},
    {"en": "tubing head", "ru": "головка насосно-компрессорных труб", "pos": "noun",
     "def_en": "A wellhead component that supports the tubing hanger and provides a means of attaching the Christmas tree to the wellhead."},
    {"en": "two-phase flow", "ru": "двухфазный поток", "pos": "noun",
     "def_en": "The simultaneous oil and free gas flow into a wellbore."},
    {"en": "two-phase separator", "ru": "двухфазный сепаратор", "pos": "noun",
     "def_en": "A vessel that separates the well fluids into gas and total liquid."},
    {"en": "air-foam", "ru": "пенообразующий агент", "pos": "noun",
     "def_en": "A light, frothy mass of fine bubbles formed in or on the surface of a liquid or from a liquid."},
    {"en": "clutch", "ru": "соединительная муфта; сцепление", "pos": "noun",
     "def_en": "Permit gradual engaging and disengaging of the equipment driven by a prime mover."},
    {"en": "coalescer", "ru": "коагулятор", "pos": "noun",
     "def_en": "A device that induces coalescence in a medium."},
    {"en": "coalescing filter", "ru": "фильтр-коалессер; нефтяная ловушка", "pos": "noun",
     "def_en": "A filter used in compressed air systems to remove oil droplets and fine particles down to 0.01 microns in size."},
    {"en": "coupling", "ru": "муфта", "pos": "noun",
     "def_en": "A device for connecting parts of machinery."},
#50
    {"en": "covalent union", "ru": "ковалентная связь", "pos": "noun",
     "def_en": "A chemical bond characterized by one or more pairs of shared electrons."},
    {"en": "gas anchor", "ru": "газовый якорь", "pos": "noun",
     "def_en": "A short section of tubing that extends down from an insert sucker-rod pump and separates gas from the oil before it enters the pump to prevent gas locking."},
    {"en": "gasometer", "ru": "газометр", "pos": "noun",
     "def_en": "A large container in which natural gas or town gas is stored near atmospheric pressure at ambient temperatures."},
    {"en": "permafrost", "ru": "вечная мерзлота", "pos": "noun",
     "def_en": "A permanently frozen layer on or under Earth's surface."},
    {"en": "bubble flow", "ru": "пузырьковый режим потока", "pos": "noun",
     "def_en": "A multiphase flow regime in pipes in which one fluid moves as small dispersed bubbles through a continuous fluid."},
    {"en": "bubble point", "ru": "точка образования пузырьков", "pos": "noun",
     "def_en": "The temperature at a given pressure or a pressure at a given temperature at the instant the first bubble of gas is formed in a given liquid."},
    {"en": "bundle", "ru": "трубный пучок", "pos": "noun",
     "def_en": "Several pipes (production or injection, gas lift) that are jointly insulated to keep production lines together."},
    {"en": "bulk boat", "ru": "нефтяной танкер", "pos": "noun",
     "def_en": "A ship designed for the bulk transport of oil or its products."},
    {"en": "bushwash", "ru": "нефтяной отстой", "pos": "noun",
     "def_en": "Sludge at the bottom of an oil tank, not collapsing without heating; or emulsion of oil and water."},
    {"en": "flow loop", "ru": "гидравлический испытательный стенд замкнутого типа", "pos": "noun",
     "def_en": "A laboratory instrument for investigating fluid flow characteristics in pipes and for studying the response of production logging instruments to this flow."},
 #60
    {"en": "water hammer", "ru": "гидравлический удар", "pos": "noun",
     "def_en": "The physical effect, often accompanied by loud banging, which is produced by pressure waves generated by a rapid change of velocity in a liquid system within the piping."},
    {"en": "cracking", "ru": "крекинг", "pos": "noun",
     "def_en": "The process of splitting a large heavy hydrocarbon molecule into smaller, lighter components."},
    {"en": "crane", "ru": "кран; подъёмный кран", "pos": "noun",
     "def_en": "A type of machine, generally equipped with a hoist rope, wire ropes or chains, and sheaves, that can be used both to lift and lower materials and to move them horizontally."},
    {"en": "crankshaft", "ru": "коленчатый вал", "pos": "noun",
     "def_en": "A shaft driven by a crank mechanism consisting of a series of cranks and crankpins to which the connecting rods of an engine are attached."},
    {"en": "adsorbed gas", "ru": "адсорбированный газ", "pos": "noun",
     "def_en": "The gas that accumulates on the surface of a solid material, such as a grain of a reservoir rock or, more particularly, the organic particles in a shale reservoir."},
    {"en": "adsorption", "ru": "адсорбция", "pos": "noun",
     "def_en": "The property of some solids and liquids to attract a liquid or a gas to their surfaces."},
    {"en": "azimuth", "ru": "азимут", "pos": "noun",
     "def_en": "The angle between the vertical projection of a line of interest onto a horizontal surface and true north or magnetic north measured in a horizontal plane, typically measured clockwise from north."},
    {"en": "nitrogen cushion", "ru": "азотная подушка", "pos": "noun",
     "def_en": "A means of allowing a precise pressure differential before opening flow from the reservoir."},
    {"en": "centrifugal pump", "ru": "центробежный насос", "pos": "noun",
     "def_en": "A type of pump commonly used in the handling and mixing oilfield fluids."},
    {"en": "three-phase separator", "ru": "трёхфазный сепаратор", "pos": "noun",
     "def_en": "A vessel with cylindrical axes that separates oil, gas, and water from the production stream."},
 #70
    {"en": "cold production", "ru": "холодная добыча", "pos": "noun",
     "def_en": "Nonthermal primary heavy oil production methods that include horizontal wells, multilateral, CHOPS, and water or gas injection."},
    {"en": "buster", "ru": "газосепаратор", "pos": "noun",
     "def_en": "A simple separator vessel that removes free or entrained gas from fluids circulated in the wellbore."},
    {"en": "slam retarder", "ru": "гаситель гидравлического удара", "pos": "noun",
     "def_en": "A device to prevent the clapper of a check valve from slamming as it closes upon flow reversal."},
    {"en": "generator", "ru": "генератор", "pos": "noun",
     "def_en": "A device that creates electrical energy to power appliances running on electricity by burning the fuel it uses for producing electricity."},
    {"en": "sealing", "ru": "герметизация", "pos": "noun",
     "def_en": "The surface-to-surface joining technique of materials using a substance of a different type that adheres to the surfaces of the two adherents to be joined, transferring the forces from one adherent to the other."},
    {"en": "accumulator", "ru": "гидравлический аккумулятор", "pos": "noun",
     "def_en": "A vessel used in a hydraulic system to store energy or, in some applications, dampen pressure fluctuations."},
    {"en": "downhole gauge", "ru": "глубинный манометр", "pos": "noun",
     "def_en": "A device used in assessing the downhole pressure under various flowing conditions, the basis of pressure transient analysis."},
    {"en": "archie rock", "ru": "петрофизически простая горная порода", "pos": "noun",
     "def_en": "A rock that is petrophysical simple."},
    {"en": "drilling rig", "ru": "буровая установка", "pos": "noun",
     "def_en": "A rig or drilling rig is a machine used to drill a wellbore."},
    {"en": "drilling contractor", "ru": "буровой подрядчик", "pos": "noun",
     "def_en": "A drilling contractor is a company that owns and operates a drilling rig."},
#80
    {"en": "acid gas", "ru": "кислый газ", "pos": "noun",
     "def_en": "A gas that can form acidic solutions when mixed with water. Hydrogen sulfide (H2S) and carbon dioxide (CO2) are the most common acid gases."},
    {"en": "acetic acid", "ru": "уксусная кислота", "pos": "noun",
     "def_en": "An organic acid used in oil and gas well-stimulation treatments as an inhibit or retarder."},
    {"en": "acid inhibitor", "ru": "кислотный ингибитор", "pos": "noun",
     "def_en": "A chemical additive used to protect wellbore components and treatment equipment from the corrosive action of an acid."},
    {"en": "gas field", "ru": "газовое месторождение", "pos": "noun",
     "def_en": "A district where natural gas is produced in commercial quantities."},
    {"en": "spacer fluid", "ru": "буферная жидкость", "pos": "noun",
     "def_en": "Any liquid used to separate one special-purpose liquid from another."},
    {"en": "crown valve", "ru": "буферная задвижка", "pos": "noun",
     "def_en": "The topmost valve on a Christmas tree that provides vertical access to the wellbore."},
    {"en": "gross production", "ru": "валовое производство", "pos": "noun",
     "def_en": "The total production of oil, gas, or water from a well or field over a specific period of time."},
    {"en": "fan", "ru": "вентилятор", "pos": "noun",
     "def_en": "A machine used to move fluid or air."},
    {"en": "duct", "ru": "воздуховод", "pos": "noun",
     "def_en": "A tube or passageway in a machine for air, liquid, cables, etc."},
    {"en": "wax", "ru": "парафин; воск", "pos": "noun",
     "def_en": "A mixture of solid hydrocarbons."},
#90
    {"en": "well flow rate", "ru": "дебит скважины", "pos": "noun",
     "def_en": "The well production or injection rate."},
    {"en": "degasser", "ru": "дегазатор", "pos": "noun",
     "def_en": "A device that removes air or gases from drilling liquids."},
    {"en": "gas blanket", "ru": "газовая подушка; защитный газовый слой", "pos": "noun",
     "def_en": "A gas phase maintained above a liquid in a vessel to protect the liquid against air contamination, to reduce the hazard of detonation, or to pressurize the liquid."},
    {"en": "butane", "ru": "бутан", "pos": "noun",
     "def_en": "One of the lightest liquid streams typically produced in a refinery."},
    {"en": "tankage", "ru": "вместимость", "pos": "noun",
     "def_en": "The capacity of all the tanks in a field."},
    {"en": "pigging", "ru": "внутренняя очистка трубопровода скребками", "pos": "noun",
     "def_en": "The act of forcing a device called a pig through a pipeline to displace or separate fluids and cleaning or inspecting the line."},
    {"en": "air-powered", "ru": "с пневматическим приводом", "pos": "adjective",
     "def_en": "Moved or worked by air pressure."},
    {"en": "roto-rabbit", "ru": "вращающийся скребок", "pos": "noun",
     "def_en": "A small plug that is passed through a pressurized pressure line to clear the line or check for obstructions."},
    {"en": "cap the well", "ru": "закрывать; запечатывать; заглушать; ликвидировать скважину", "pos": "verb",
     "def_en": "Remain control of a blowout well by installing and closing a valve on the wellhead."},
    {"en": "all-welded", "ru": "цельносварный", "pos": "adjective",
     "def_en": "Completely welded and cannot be disassembled and repaired in the field."}
#100
]

SUBJECT_FIELD = "Нефтегазовая отрасль: добыча и переработка"
TBX_PATH = "variant_7.tbx"


def build_tbx(terms: Sequence[Dict[str, str]], path: str,
              subject_field: str = SUBJECT_FIELD) -> str:
    """Сгенерировать валидный TBX-файл (TBX-Core v2, martif).

    Args:
        terms: последовательность словарей с ключами ``en``, ``ru``, ``pos``, ``def_en``.
        path: путь к создаваемому файлу.
        subject_field: предметная область, записывается в ``descrip[subjectField]``.

    Returns:
        Путь к созданному файлу.
    """
    martif = ET.Element("martif", {"type": "TBX", XML_LANG: "en"})

    header = ET.SubElement(martif, "martifHeader")
    file_desc = ET.SubElement(header, "fileDesc")
    title_stmt = ET.SubElement(file_desc, "titleStmt")
    ET.SubElement(title_stmt, "title").text = f"Glossary EN-RU: {subject_field}"
    source_desc = ET.SubElement(file_desc, "sourceDesc")
    ET.SubElement(source_desc, "p").text = "Lab 01, variant 25, generated automatically"
    encoding_desc = ET.SubElement(header, "encodingDesc")
    ET.SubElement(encoding_desc, "p", {"type": "XCSURI"}).text = "TBXXCSV02.xcs"

    body = ET.SubElement(ET.SubElement(martif, "text"), "body")

    for index, item in enumerate(terms, start=1):
        entry = ET.SubElement(body, "termEntry", {"id": f"tid-{index:03d}"})
        ET.SubElement(entry, "descrip", {"type": "subjectField"}).text = subject_field

        # --- английская секция: краткая форма <tig> ---
        lang_en = ET.SubElement(entry, "langSet", {XML_LANG: "en"})
        tig = ET.SubElement(lang_en, "tig")
        ET.SubElement(tig, "term").text = item["en"]
        ET.SubElement(tig, "termNote", {"type": "partOfSpeech"}).text = item["pos"]
        ET.SubElement(tig, "descrip", {"type": "definition"}).text = item["def_en"]

        # --- русская секция: расширенная форма <ntig><termGrp> ---
        lang_ru = ET.SubElement(entry, "langSet", {XML_LANG: "ru"})
        term_grp = ET.SubElement(ET.SubElement(lang_ru, "ntig"), "termGrp")
        ET.SubElement(term_grp, "term").text = item["ru"]
        ET.SubElement(term_grp, "termNote", {"type": "partOfSpeech"}).text = item["pos"]

    indent_tree(martif)
    xml_body = ET.tostring(martif, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">\n')
        handler.write(xml_body)
    return path


build_tbx(TERMS, TBX_PATH)
print(f"Создан {TBX_PATH}: {os.path.getsize(TBX_PATH)} байт, терминов: {len(TERMS)}\n")
print("--- первые 26 строк файла ---")
with open(TBX_PATH, encoding="utf-8") as handler:
    for line in list(handler)[:26]:
        print(line.rstrip())

Создан variant_7.tbx: 76783 байт, терминов: 100

--- первые 26 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">
<martif type="TBX" xml:lang="en">
  <martifHeader>
    <fileDesc>
      <titleStmt>
        <title>Glossary EN-RU: Нефтегазовая отрасль: добыча и переработка</title>
      </titleStmt>
      <sourceDesc>
        <p>Lab 01, variant 25, generated automatically</p>
      </sourceDesc>
    </fileDesc>
    <encodingDesc>
      <p type="XCSURI">TBXXCSV02.xcs</p>
    </encodingDesc>
  </martifHeader>
  <text>
    <body>
      <termEntry id="tid-001">
        <descrip type="subjectField">Нефтегазовая отрасль: добыча и переработка</descrip>
        <langSet xml:lang="en">
          <tig>
            <term>amplitude</term>
            <termNote type="partOfSpeech">noun</termNote>
            <descrip type="definition">Difference between the maximum displacement of a wave and the point of no displacement or the null point.</descrip>


In [5]:
PARALLEL_CORPUS: List[Tuple[str, str]] = [
    ("Sedimentary rocks generated as a result of organic and inorganic matter settling on bottoms of water basins and surface of continents.",
    "Осадочные породы образовались в результате осаждения органических и неорганических веществ на дне водных бассейнов и поверхности материков."),
    ("Seismic exploration goal is to obtain information on rocks on the basis of observed time of wave arrival, amplitude, frequency and form of signal.",
    "Цель сейсморазведки состоит в получении информации о породах на основе наблюдаемого времени вступления, амплитуды, частоты и формы сигнала волны."),
    ("Rocks saturated with oil or gas, have lower density than the same rocks containing water.",
    "Породы, насыщенные нефтью или газом, имеют меньшую плотность, чем те же породы, содержащие воду."),
    ("Field exploration stage includes implementation of geological and seismic exploration operations, drilling of wildcats and then exploratory wells.",
    "Этап разведки месторождения включает проведение геологических и сейсморазведочных работ, бурение поисковых, а затем разведочных скважин."),
    ("The inorganic theory assumes that carbon and hydrogen were combining due to high pressure and temperature deep under ground and thus generated oil and gas.",
    "Неорганическая теория предполагает, что углерод и водород соединялись в результате высокого давления и температуры глубоко под землёй и таким образом образовали нефть и газ."),
    ("Oil and gas blowout may result in damage of drilling equipment, an explosion, fire or injuries of maintenance personnel.",
    "Выброс нефти и газа может привести к повреждению бурового оборудования, взрыву, пожару или травмам обслуживающего персонала."),
    ("Special equipment called a blowout preventer must be installed on each well being drilled in order to prevent blowouts.",
    "На каждой бурящейся скважине должно быть установлено специальное оборудование, называемое противовыбросовым превентором, для предотвращения выбросов."),
    ("Blowout preventers are devices installed on the casing head to provide for protection against potential well blowouts.",
    "Противовыбросовые превенторы — это устройства, устанавливаемые на обсадной головке для защиты от возможных выбросов из скважины."),
    ("Christmas tree is the main component of wellhead equipment.",
    "Фонтанная арматура является основным компонентом оборудования устья скважины."),
    ("Directional drilling is a type of drilling, in which the wellbore is deviated in relation to stratification.",
    "Наклонно-направленное бурение — это вид бурения, при котором ствол скважины искривляется относительно напластования."),
#10
    ("Production wells are spudded to recover oil and gas from subsurface reservoirs.",
     "Эксплуатационные скважины бурятся для извлечения нефти и газа из подземных коллекторов."),
    ("Asphaltenes exist as colloidal suspensions in the oil.",
     "Асфальтены существуют в нефти в виде коллоидных суспензий."),
    ("The bit is on the bottom of the drillstring, usually as part of a rotational motion.",
     "Долото находится в нижней части бурильной колонны и обычно участвует во вращательном движении."),
    ("If reservoir fluids flow into another formation and do not flow to the surface, the result is an underground blowout.",
     "Если пластовые флюиды перетекают в другой пласт и не выходят на поверхность, возникает подземный выброс."),
    ("During the appraisal, delineation wells might be drilled to determine the size of the oil or gas field and how to develop it most efficiently.",
     "На этапе оценки месторождения могут бурить оценочные скважины, чтобы определить размеры нефтяного или газового месторождения и наиболее эффективный способ его разработки."),
    ("Drillers are often concerned about aquifers and must take special precautions in the design and execution of the well plan to protect freshwater aquifers from contamination by wellbore and cause well control problems.",
     "Буровики часто уделяют особое внимание водоносным горизонтам и должны принимать специальные меры при проектировании и выполнении плана строительства скважины, чтобы защитить пресноводные водоносные горизонты от загрязнения из скважины и не допустить проблем с контролем скважины."),
    ("The burner is attached to a heating appliance, water heater, or boiler to heat air or water.",
     "Горелка присоединяется к отопительному устройству, водонагревателю или котлу для нагрева воздуха или воды."),
    ("Air drilling uses compressed air to remove rock cuttings from the borehole.",
     "При бурении с использованием воздуха сжатый воздух используется для удаления выбуренной породы из скважины."),
    ("Asphalt is used in road construction and can be produced from petroleum.",
     "Асфальт используется в дорожном строительстве и может производиться из нефти."),
    ("The borehole must be cleaned before the casing is installed.",
     "Ствол скважины необходимо очистить перед установкой обсадной колонны."),
 #20
    ("A beam pump lifts oil from the reservoir to the surface using a reciprocating sucker-rod system.",
    "Глубинный насос штангового типа поднимает нефть из пласта на поверхность с помощью возвратно-поступательной системы насосных штанг."),
    ("The ball valve regulates the flow of fluid through the pipeline.",
     "Шаровой клапан регулирует поток жидкости через трубопровод."),
    ("The oil field produces 5,000 barrels of crude oil per day.",
     "Месторождение добывает 5 000 баррелей сырой нефти в сутки."),
    ("Excessive backpressure can reduce the production rate of an oil well.",
     "Чрезмерное противодавление может снизить дебит нефтяной скважины."),
    ("Blinding the pipeline before maintenance helps isolate the equipment from the process fluids.",
     "Установка заглушек на трубопроводе перед техническим обслуживанием помогает изолировать оборудование от технологических жидкостей."),
    ("The casing head supports the casing string and provides a connection for wellhead equipment.",
     "Обсадная головка поддерживает обсадную колонну и обеспечивает соединение с устьевым оборудованием."),
    ("Gas compressibility changes with pressure and temperature.",
     "Сжимаемость газа изменяется в зависимости от давления и температуры."),
    ("Cementing seals the space between the casing and the borehole wall.",
     "Цементирование герметизирует пространство между обсадной колонной и стенкой скважины."),
    ("Chokes are available in several configurations for both fixed and adjustable modes of operation.",
    "Дроссели доступны в нескольких конфигурациях как для фиксированного, так и для регулируемого режима работы."),
    ("The condensate is not only generated into the reservoir but also formed when liquid drops out or condenses from a gas stream in pipelines or surface facilities.",
    "Конденсат образуется не только в пласте, но и при выпадении или конденсации жидкости из газового потока в трубопроводах или наземных сооружениях."),
#30
    ("Condensate gas mainly consists of propane, butane, pentane, and heavier hydrocarbon fractions.",
    "Газовый конденсат в основном состоит из пропана, бутана, пентана и более тяжёлых углеводородных фракций."),
    ("The inhibitor does not entirely stop the corrosion reaction but eliminates more than 99% of the metal losses that would occur if the inhibitor were absent.",
    "Ингибитор не останавливает реакцию коррозии полностью, но устраняет более 99% потерь металла, которые произошли бы в его отсутствие."),
    ("Corrosion-resistant alloys can be formulated for a wide range of aggressive wellbore conditions.",
    "Коррозионностойкие сплавы могут разрабатываться для широкого спектра агрессивных условий в стволе скважины."),
    ("The formula for density is the mass of an object divided by its volume.",
    "Формула плотности — это масса объекта, делённая на его объём."),
    ("Fluid dehydration is needed to prevent corrosion and free-water accumulation in the low points of a pipeline.",
    "Обезвоживание жидкости необходимо для предотвращения коррозии и скопления свободной воды в нижних точках трубопровода."),
    ("Demulsification helps separate water from crude oil by breaking the emulsion.",
    "Деэмульгация помогает отделить воду от сырой нефти путём разрушения эмульсии."),
    ("Engineers studied how the fluid behaved in the dense phase during gas processing.",
    "Инженеры изучали поведение флюида в плотной фазе во время переработки газа."),
    ("The derrick floor is where the rig crew can meet, take breaks, and take refuge from the elements during idle times.",
    "Рабочая площадка буровой установки — это место, где бригада может собираться, отдыхать и укрываться от непогоды во время простоев."),
    ("The desalter removes salts and water from crude oil before refining.",
    "Обессоливающая установка удаляет соли и воду из сырой нефти перед переработкой."),
    ("Rapid decompression can cause a sudden decrease in pressure inside the equipment.",
    "Быстрая декомпрессия может вызвать резкое снижение давления внутри оборудования."),
 #40
    ("Oil and gas exploration encompasses the processes and methods of locating potential oil and gas drilling and extraction sites.",
    "Геологоразведочные работы в нефтегазовой отрасли включают процессы и методы поиска потенциальных мест для бурения и добычи нефти и газа."),
    ("Hydraulic fracturing causes the formation to crack open, creating passages for the reservoir hydrocarbons to flow more easily into the wellbore.",
    "Гидравлический разрыв пласта вызывает растрескивание породы, создавая каналы, по которым углеводороды из пласта легче поступают в ствол скважины."),
    ("The operator reduced operating costs to keep the marginal well in production.",
    "Оператор сократил эксплуатационные расходы, чтобы продолжить добычу на низкорентабельной скважине."),
    ("Sedimentary basins vary from bowl-shaped to elongated troughs.",
    "Осадочные бассейны бывают от чашеобразных до вытянутых впадин."),
    ("Hydrostatic pressure is a column of fluid that causes the force per unit area.",
    "Гидростатическое давление создаётся столбом жидкости, который оказывает силу на единицу площади."),
    ("As drilling fluid is pumped through the positive displacement motor, it converts the hydraulic power of the fluid into mechanical power to cause the bit to rotate.",
    "Когда буровой раствор прокачивается через гидравлический забойный двигатель, он преобразует гидравлическую энергию жидкости в механическую энергию, заставляя долото вращаться."),
    ("Rotary drilling is continuous because cuttings are removed as drilling fluids circulate through the bit and up the wellbore to the surface.",
    "Роторное бурение является непрерывным процессом, поскольку шлам удаляется, когда буровой раствор циркулирует через долото и поднимается по стволу скважины на поверхность."),
    ("Two-phase flow is thought to occur through common pore spaces, where the fluids flow simultaneously.",
    "Предполагается, что двухфазный поток проходит через общие поровые пространства, где флюиды движутся одновременно."),
    ("A two-phase separator scrubs free liquids from your vent gas stream before entering your flare system's relief and blow-down lines.",
    "Двухфазный сепаратор удаляет свободные жидкости из потока выпускаемого газа перед поступлением в линии сброса и продувки факельной установки."),
    ("Coalescers are primarily used to separate emulsions into components via various processes, operating in reverse to an emulsifier.",
    "Коагуляторы преимущественно используются для разделения эмульсий на компоненты с помощью различных процессов, действуя противоположно эмульгатору."),
 #50
    ("Coalescing filters are used in compressed air systems to remove oil droplets and fine particles down to 0.01 microns in size.",
    "Фильтры-коалессеры используются в системах сжатого воздуха для удаления капель масла и мелких частиц размером до 0,01 микрона."),
    ("Hydrocarbons are covalent compounds composed of carbon and hydrogen atoms.",
    "Углеводороды представляют собой соединения с ковалентными связями, состоящие из атомов углерода и водорода."),
    ("Permafrost consists of soil, gravel, and sand, usually bound together by ice.",
    "Вечная мерзлота состоит из почвы, гравия и песка, обычно связанных между собой льдом."),
    ("In bubble flow, the relative velocity of the bubbles depends mainly on the difference in density between the two fluids.",
    "При пузырьковом режиме потока относительная скорость пузырьков зависит главным образом от разницы в плотности двух жидкостей."),
    ("The bundle minimizes heat transfer and avoids hydrate or wax deposition that could plug the pipelines.",
    "Трубный пучок сводит к минимуму теплопередачу и предотвращает отложение гидратов или парафина, которые могут закупорить трубопроводы."),
    ("Flow loops are essential for the study of multiphase flow and the development of new production logging measurements.",
    "Гидравлические испытательные стенды замкнутого типа необходимы для изучения многофазных потоков и разработки новых методов промыслового каротажа."),
    ("The gas plant uses sweetening before sending the gas to the pipeline.",
    "На газоперерабатывающем предприятии проводят десульфурацию перед подачей газа в трубопровод."),
    ("The crew inspected the tubing head before installing the Christmas tree.",
    "Бригада осмотрела головку насосно-компрессорных труб перед установкой фонтанной арматуры."),
    ("Air-foam was selected for the treatment because the operation required a lightweight fluid system.",
    "Для обработки выбрали пенообразующий агент, поскольку операция требовала облегчённой жидкостной системы."),
    ("The mechanic engaged the clutch to start the auxiliary equipment.",
    "Механик включил сцепление, чтобы запустить вспомогательное оборудование."),
  #60
    ("The coupling loosened during operation, so the maintenance team stopped the pump.",
    "Во время работы муфта ослабла, поэтому ремонтная бригада остановила насос."),
    ("The gas anchor was checked after the pump's performance became unstable.",
    "Газовый якорь проверили после того, как работа насоса стала нестабильной."),
    ("The gasometer supplied stored gas to the facility during maintenance of the main line.",
    "Во время обслуживания основной линии газометр подавал на объект хранящийся в нём газ."),
    ("The engineers adjusted the test conditions to investigate bubble flow at a higher liquid rate.",
    "Инженеры изменили условия испытания, чтобы исследовать пузырьковый режим потока при более высокой скорости движения жидкости."),
    ("The laboratory measured the bubble point of the sample before conducting the next pressure test.",
    "Лаборатория измерила точку образования пузырьков в образце перед следующим испытанием под давлением."),
    ("The bulk boat was scheduled to deliver crude oil to the export terminal.",
    "Нефтяной танкер должен был доставить сырую нефть в экспортный терминал."),
    ("The tank was heated to make the bushwash easier to remove.",
    "Резервуар нагрели, чтобы облегчить удаление нефтяного отстоя."),
    ("A water hammer can occur when the flow of liquid in a pipeline changes rapidly.",
    "Гидравлический удар может возникнуть при резком изменении потока жидкости в трубопроводе."),
    ("Butane is typically produced in a refinery and may be obtained through the cracking of petroleum fractions.",
    "Бутан обычно получают на нефтеперерабатывающем заводе, в том числе путём крекинга нефтяных фракций."),
    ("The drilling rig includes cranes and other equipment used for lifting and lowering materials.",
    "Буровая установка включает краны и другое оборудование, используемое для подъёма и опускания материалов."),
  #70
    ("Crankshaft is able to perform a conversion between reciprocating motion and rotational motion.",
    "Коленчатый вал преобразует возвратно-поступательное движение поршней в крутящий момент."),
    ("Adsorbed gas can accumulate on organic particles in a shale reservoir.",
    "Адсорбированный газ может накапливаться на органических частицах в сланцевом коллекторе."),
    ("Adsorption can be used to gather liquid hydrocarbons from a natural gas stream.",
    "Адсорбцию можно использовать для сбора жидких углеводородов из потока природного газа."),
    ("The azimuth is generally quoted in degrees from 0 to 359.",
    "Азимут обычно указывается в градусах от 0 до 359."),
    ("Once flow begins, the nitrogen cushion pressure can be easily and safely bled down to flow formation fluids under high control.",
    "После начала потока давление азотной подушки можно легко и безопасно снижать, обеспечивая строгий контроль притока пластовых флюидов."),
    ("The centrifugal pump maintained the required flow rate while the oilfield fluids were being transferred.",
    "Центробежный насос поддерживал необходимый расход при перекачке нефтепромысловых жидкостей."),
    ("The horizontal separator can be a two-phase or three-phase separator.",
    "Горизонтальный сепаратор может быть двухфазным или трехфазным сепаратором."),
    ("Cold production relies on nonthermal methods to recover heavy oil from a reservoir.",
    "При холодной добыче тяжёлую нефть извлекают из пласта нетермическими методами."),
    ("The gas buster typically comprises a vessel containing a series of baffles with a liquid exit at the bottom and a gas-vent line at the top.",
    "Газосепаратор обычно представляет собой сосуд с рядом перегородок, выходом жидкости внизу и газоотводной линией вверху."),
    ("The slam retarder prevents the check-valve clapper from slamming shut when the flow reverses.",
    "Гаситель гидравлического удара предотвращает резкое захлопывание тарелки обратного клапана при изменении направления потока."),
  #80
    ("The generator supplies electrical power to equipment operating at the drilling site.",
    "Генератор обеспечивает электроэнергией оборудование, работающее на буровой площадке."),
    ("The substance used for sealing is known as a sealant or sealer.",
    "Вещество, используемое для герметизации, известно как герметик."),
    ("Closing unit – a generic term given to the hydraulic power pack and accumulators used to control the blowout preventers on a drilling or workover rig.",
    "Закрытие блока – это общий термин для гидравлической силовой установки и гидроаккумуляторов, используемым для управления противовыбросовыми превенторами на буровой или ремонтной установке."),
    ("The downhole gauge measures pressure under different flowing conditions.",
    "Глубинный манометр измеряет давление при различных условиях потока."),
    ("Rocks that follow the porosity-resistivity trends described by Archie’s equation are defined as the Archie rocks.",
    "Породы, соответствующие зависимостям пористости и удельного электрического сопротивления, описываемым уравнением Арчи, называют породами Арчи."),
    ("Major components of the drilling rig include the mud tanks, mud pumps, derrick, drawworks, and drillstring.",
    "К основным компонентам буровой установки относятся ёмкости для бурового раствора, буровые насосы, вышка, лебёдки и бурильная колонна."),
    ("The drilling foreman is the location supervisor for the drilling contractor.",
    "Буровой мастер - руководитель работ на объекте со стороны бурового подрядчика."),
    ("The water is removed while the acid gas can be flared or further treated in a sulfur recovery unit to separate elemental sulfur.",
    "Вода удаляется, в то время как кислый газ может сжигаться в факеле или подвергаться дальнейшей обработке в установке для извлечения серы для отделения элементарной серы."),
    ("In most cases, acetic acid is used with hydrochloric acid and other additives.",
    "В большинстве случаев уксусная кислота используется с соляной кислотой и другими кислотными добавками."),
    ("The engineer added an acid inhibitor to the treatment fluid before pumping it into the well.",
    "Инженер добавил кислотный ингибитор в жидкость для обработки перед её закачкой в скважину."),
#90
    ("The company plans to develop a new gas field in the region.",
    "Компания планирует освоить новое газовое месторождение в этом регионе."),
    ("The crew pumped a spacer fluid into the well before introducing the cement slurry.",
    "Бригада закачала буферную жидкость в скважину перед подачей цементного раствора."),
    ("The technician inspected the crown valve before opening the well for production.",
    "Техник осмотрел буферную задвижку перед тем, как открыть скважину для добычи."),
    ("The engineer compared the field's gross production for January and February.",
    "Инженер сравнил валовую добычу месторождения за январь и февраль."),
    ("A fan is operated through a motor via electricity which rotates the blades attached to a shaft..",
    "Вентилятор приводится в действие электродвигателем, который вращает лопасти, прикрепленные к валу."),
    ("The maintenance team found a damaged duct and replaced it during the equipment inspection.",
    "Во время осмотра оборудования ремонтная бригада обнаружила повреждённый воздуховод и заменила его."),
    ("Cloud point is the temperature at which wax crystals first start to form in crude oil.",
    "Температура начала парафинообразования – это температура, при которой в сырой нефти впервые начинают формироваться кристаллы парафина."),
    ("The operator recorded the well flow rate every hour to monitor changes in production.",
    "Оператор каждый час записывал дебит скважины, чтобы отслеживать изменения добычи."),
    ("The drilling crew used the degasser when gas began accumulating in the drilling fluid.",
    "Буровая бригада использовала дегазатор, когда в буровом растворе начал накапливаться газ."),
    ("The operator checked the gas blanket above the liquid before carrying out maintenance on the storage tank.",
    "Оператор проверил газовую подушку над жидкостью перед проведением технического обслуживания резервуара."),
#100
    ("Butane is typically either blended into gasoline or LPG or sold directly as a finished product.",
    "Бутан обычно смешивают с бензином или сжиженным углеводородным газом либо продают непосредственно как готовый продукт."),
    ("The tankage of the field determines the total capacity available for storing its production.",
    "Вместимость резервуаров месторождения определяет общую ёмкость, доступную для хранения добытой продукции."),
    ("Pigging is used to clean or inspect the pipeline.",
    "Внутреннюю очистку трубопровода скребками проводят для очистки или осмотра трубопровода."),
    ("The air-powered tool operates using compressed air.",
    "Инструмент с пневматическим приводом работает на сжатом воздухе."),
    ("The crew sent a roto-rabbit through the pipeline after the inspection revealed a possible obstruction.",
    "После того как проверка выявила возможное препятствие, бригада пропустила вращающийся скребок через трубопровод."),
    ("The crew had to cap the well to regain control after the blowout.",
    "После выброса бригаде пришлось заглушить скважину, чтобы восстановить контроль над ней."),
    ("The pipeline section uses an all-welded design.",
    "На этом участке трубопровода используется цельносварная конструкция.")
]

TMX_PATH = "variant_7.tmx"


def build_tmx(pairs: Sequence[Tuple[str, str]], path: str,
              src_lang: str = "en", tgt_lang: str = "ru",
              domain: str = "Aerospace / UAS") -> str:
    """Сгенерировать валидную память переводов в формате TMX 1.4b.

    Args:
        pairs: последовательность кортежей ``(текст_источника, текст_перевода)``.
        path: путь к создаваемому файлу.
        src_lang: код языка оригинала (атрибут ``srclang`` заголовка).
        tgt_lang: код языка перевода.
        domain: значение свойства ``prop[type="domain"]``.

    Returns:
        Путь к созданному файлу.
    """
    created = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    tmx = ET.Element("tmx", {"version": "1.4"})
    ET.SubElement(tmx, "header", {
        "creationtool": "lab01-generator",
        "creationtoolversion": "1.0",
        "segtype": "sentence",
        "o-tmf": "TMX",
        "adminlang": "en",
        "srclang": src_lang,
        "datatype": "plaintext",
        "creationdate": created,
    })
    body = ET.SubElement(tmx, "body")

    for index, (source, target) in enumerate(pairs, start=1):
        tu = ET.SubElement(body, "tu", {"tuid": str(index), "creationdate": created})
        ET.SubElement(tu, "prop", {"type": "domain"}).text = domain
        for lang, text in ((src_lang, source), (tgt_lang, target)):
            tuv = ET.SubElement(tu, "tuv", {XML_LANG: lang})
            ET.SubElement(tuv, "seg").text = text

    indent_tree(tmx)
    xml_body = ET.tostring(tmx, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE tmx SYSTEM "tmx14.dtd">\n')
        handler.write(xml_body)
    return path


build_tmx(PARALLEL_CORPUS, TMX_PATH)
print(f"Создан {TMX_PATH}: {os.path.getsize(TMX_PATH)} байт, единиц перевода: {len(PARALLEL_CORPUS)}\n")
print("--- первые 20 строк файла ---")
with open(TMX_PATH, encoding="utf-8") as handler:
    for line in list(handler)[:20]:
        print(line.rstrip())

Создан variant_7.tmx: 55683 байт, единиц перевода: 107

--- первые 20 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE tmx SYSTEM "tmx14.dtd">
<tmx version="1.4">
  <header creationtool="lab01-generator" creationtoolversion="1.0" segtype="sentence" o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext" creationdate="20261018T164858Z"/>
  <body>
    <tu tuid="1" creationdate="20261018T164858Z">
      <prop type="domain">Aerospace / UAS</prop>
      <tuv xml:lang="en">
        <seg>Sedimentary rocks generated as a result of organic and inorganic matter settling on bottoms of water basins and surface of continents.</seg>
      </tuv>
      <tuv xml:lang="ru">
        <seg>Осадочные породы образовались в результате осаждения органических и неорганических веществ на дне водных бассейнов и поверхности материков.</seg>
      </tuv>
    </tu>
    <tu tuid="2" creationdate="20261018T164858Z">
      <prop type="domain">Aerospace / UAS</prop>
      <tuv xml:lang="en">
      

## Блок 4. Задание — реализуйте четыре функции

В этом блоке разрабатываются функции для обработки терминологического словаря TBX и памяти переводов TMX, а также для поиска совпадений с их содержимым. Данные извлекаются из XML-файлов и преобразуются в таблицы pandas.DataFrame, после чего реализуется алгоритм определения степени сходства текстовых сегментов. Это позволяет автоматизировать поиск переводов и терминов, а также определять тип найденного совпадения.

In [10]:
def _parse_xml(path: str):
    """Безопасно разобрать XML-файл (без загрузки внешних DTD и сущностей)."""
    if LXML_AVAILABLE:
        parser = ET.XMLParser(resolve_entities=False, load_dtd=False, no_network=True)
        return ET.parse(path, parser)
    return ET.parse(path)


def validate_tbx(file_path: str) -> Dict[str, Any]:
    """Проверить структурную корректность TBX-файла.

    Проверяются: имя корневого элемента, наличие ``martifHeader``/``text``/``body``,
    наличие хотя бы двух ``langSet`` с непустым ``xml:lang`` в каждом ``termEntry``
    и наличие непустого ``<term>`` в каждой языковой секции.

    Args:
        file_path: путь к ``.tbx``-файлу.

    Returns:
        Словарь-отчёт с ключами ``valid`` (bool), ``errors`` (list[str]),
        ``entries`` (int), ``languages`` (list[str]).
    """
    errors: List[str] = []
    try:
        tree = _parse_xml(file_path)
    except Exception as exc:  # noqa: BLE001 - отчёт вместо аварийного завершения
        return {"valid": False, "errors": [f"XML не разобран: {exc}"],
                "entries": 0, "languages": []}

    root = tree.getroot()
    if root.tag != "martif":
        errors.append(f"Корневой элемент '{root.tag}', ожидается 'martif'")
    if root.find("martifHeader") is None:
        errors.append("Отсутствует секция <martifHeader>")

    body = root.find("./text/body")
    if body is None:
        errors.append("Отсутствует секция <text><body>")
        return {"valid": False, "errors": errors, "entries": 0, "languages": []}

    entries = body.findall("termEntry")
    if not entries:
        errors.append("В <body> нет ни одного <termEntry>")

    languages: List[str] = []
    for entry in entries:
        entry_id = entry.get("id", "<без id>")
        lang_sets = entry.findall("langSet")
        if len(lang_sets) < 2:
            errors.append(f"{entry_id}: языковых секций {len(lang_sets)}, нужно >= 2")
        for lang_set in lang_sets:
            lang = lang_set.get(XML_LANG)
            if not lang:
                errors.append(f"{entry_id}: у <langSet> нет атрибута xml:lang")
                continue
            if lang not in languages:
                languages.append(lang)
            terms = [t for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                errors.append(f"{entry_id}/{lang}: нет непустого <term>")

    return {"valid": not errors, "errors": errors,
            "entries": len(entries), "languages": languages}


def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминов.

    Args:
        file_path: путь к ``.tbx``-файлу.
        src_lang: код языка исходных терминов.
        tgt_lang: код языка переводных эквивалентов.

    Returns:
        ``pd.DataFrame`` с колонками ``entry_id``, ``source_term``, ``target_term``,
        ``pos``, ``definition``, ``subject_field``.
    """
    root = _parse_xml(file_path).getroot()
    rows: List[Dict[str, str]] = []

    for entry in root.iter("termEntry"):
        subject = ""
        descrip = entry.find("descrip[@type='subjectField']")
        if descrip is not None and descrip.text:
            subject = descrip.text.strip()

        by_lang: Dict[str, Dict[str, str]] = {}
        for lang_set in entry.findall("langSet"):
            lang = (lang_set.get(XML_LANG) or "").split("-")[0].lower()
            terms = [t.text.strip() for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                continue
            note = lang_set.find(".//termNote[@type='partOfSpeech']")
            definition = lang_set.find(".//descrip[@type='definition']")
            by_lang[lang] = {
                "term": terms[0],
                "pos": (note.text or "").strip() if note is not None else "",
                "definition": (definition.text or "").strip() if definition is not None else "",
            }

        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({
                "entry_id": entry.get("id", ""),
                "source_term": by_lang[src_lang]["term"],
                "target_term": by_lang[tgt_lang]["term"],
                "pos": by_lang[src_lang]["pos"] or by_lang[tgt_lang]["pos"],
                "definition": by_lang[src_lang]["definition"],
                "subject_field": subject,
            })

    return pd.DataFrame(rows, columns=["entry_id", "source_term", "target_term",
                                       "pos", "definition", "subject_field"])

def _seg_text(tuv) -> str:
    """Достать текст сегмента, игнорируя инлайновые теги TMX (ph/bpt/ept/it)."""
    seg = tuv.find("seg")
    if seg is None:
        return ""
    return re.sub(r"\s+", " ", "".join(seg.itertext())).strip()


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов.

    Args:
        file_path: путь к ``.tmx``-файлу.
        src_lang: код исходного языка.
        tgt_lang: код языка перевода.

    Returns:
        ``pd.DataFrame`` с колонками ``tuid``, ``source_segment``, ``target_segment``,
        ``domain``, ``src_len``. Строки без одного из языков отбрасываются.
    """
    root = _parse_xml(file_path).getroot()
    rows: List[Dict[str, Any]] = []

    for order, tu in enumerate(root.iter("tu"), start=1):
        segments: Dict[str, str] = {}
        for tuv in tu.findall("tuv"):
            lang = (tuv.get(XML_LANG) or tuv.get("lang") or "").split("-")[0].lower()
            text = _seg_text(tuv)
            if lang and text:
                segments[lang] = text

        if src_lang not in segments or tgt_lang not in segments:
            continue

        domain_node = tu.find("prop[@type='domain']")
        rows.append({
            "tuid": tu.get("tuid", str(order)),
            "source_segment": segments[src_lang],
            "target_segment": segments[tgt_lang],
            "domain": (domain_node.text or "").strip() if domain_node is not None else "",
            "src_len": len(segments[src_lang]),
        })

    frame = pd.DataFrame(rows, columns=["tuid", "source_segment", "target_segment",
                                        "domain", "src_len"])
    # предыдущий сегмент нужен для контекстного сопоставления (ICE)
    frame["prev_source"] = frame["source_segment"].shift(1).fillna("")
    return frame

def similarity(a: str, b: str) -> Dict[str, float]:
    """Вернуть словарь метрик сходства: ``lev``, ``token_sort``, ``score``."""
    norm_a, norm_b = normalize(a), normalize(b)
    lev = levenshtein_ratio(norm_a, norm_b)
    tsr = token_sort_ratio(a, b)
    return {"lev": lev, "token_sort": tsr, "score": max(lev, tsr)}


def classify_match(score: float, context_matched: bool) -> str:
    """Определить тип совпадения по проценту и признаку совпадения контекста."""
    if score >= 100.0:
        return "ICE (101%)" if context_matched else "Exact (100%)"
    if score >= 95.0:
        return "High Fuzzy"
    if score >= 75.0:
        return "Fuzzy"
    return "Low Fuzzy"


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame,
                          threshold: float = 70.0,
                          context_before: Optional[str] = None,
                          top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос.

    Args:
        source_query: сегмент, который нужно перевести.
        tm_df: таблица памяти переводов (результат :func:`parse_tmx`).
        threshold: минимальный процент сходства для попадания в выдачу.
        context_before: предыдущий сегмент документа; при совпадении с
            ``prev_source`` 100-процентное совпадение повышается до ICE.
        top_n: максимальное число возвращаемых кандидатов.

    Returns:
        Список словарей, отсортированный по убыванию ``score``; ключи:
        ``tuid``, ``source_segment``, ``target_segment``, ``score``,
        ``lev_ratio``, ``token_sort_ratio``, ``match_type``.
    """
    if tm_df.empty:
        return []

    results: List[Dict[str, Any]] = []
    for _, row in tm_df.iterrows():
        metrics = similarity(source_query, row["source_segment"])
        if metrics["score"] < threshold:
            continue
        context_matched = bool(
            context_before is not None
            and normalize(context_before) == normalize(str(row.get("prev_source", "")))
        )
        results.append({
            "tuid": row["tuid"],
            "source_segment": row["source_segment"],
            "target_segment": row["target_segment"],
            "score": metrics["score"],
            "lev_ratio": metrics["lev"],
            "token_sort_ratio": metrics["token_sort"],
            "match_type": classify_match(metrics["score"], context_matched),
        })

    results.sort(key=lambda item: (item["score"], item["tuid"]), reverse=True)
    return results[:top_n]

def apply_term_glossary(text: str, glossary_df: pd.DataFrame,
                        template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария.

    Args:
        text: исходный сегмент на языке оригинала.
        glossary_df: таблица глоссария (результат :func:`parse_tbx`) с колонками
            ``source_term`` и ``target_term``.
        template: шаблон подстановки; поля ``{source}`` и ``{target}``.

    Returns:
        Текст, в котором найденные термины заменены по шаблону. Порядок слов и
        регистр исходного текста сохраняются.
    """
    if glossary_df.empty or not text:
        return text

    pairs = sorted(
        zip(glossary_df["source_term"], glossary_df["target_term"]),
        key=lambda pair: len(pair[0]), reverse=True,
    )

    protected: List[str] = []

    def _store(replacement: str) -> str:
        protected.append(replacement)
        return f"\x00{len(protected) - 1}\x00"

    result = text
    for source_term, target_term in pairs:
        pattern = re.compile(rf"(?<!\w){re.escape(source_term)}(?!\w)", re.IGNORECASE)
        result = pattern.sub(
            lambda match: _store(template.format(source=match.group(0), target=target_term)),
            result,
        )

    return re.sub(r"\x00(\d+)\x00", lambda m: protected[int(m.group(1))], result)

### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [9]:
_PUNCT_RE = re.compile(r"[^\w\s]", re.UNICODE)

def normalize(text: str) -> str:
    """Привести строку к нормальной форме: NFKC, нижний регистр, без пунктуации."""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    text = _PUNCT_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein_ratio(a: str, b: str) -> float:
    """Процент сходства двух строк на основе расстояния Левенштейна (0–100)."""
    if not a and not b:
        return 100.0
    longest = max(len(a), len(b))
    return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)


def token_sort_ratio(a: str, b: str) -> float:
    """Процент сходства при игнорировании порядка слов (0–100)."""
    sorted_a = " ".join(sorted(normalize(a).split()))
    sorted_b = " ".join(sorted(normalize(b).split()))
    return levenshtein_ratio(sorted_a, sorted_b)

## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [11]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

Тест 1 (метрики сходства) — OK


In [12]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("demo.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 3, f"Ожидается 3 термина, получено {len(glossary_df)}"
for column in ["entry_id", "source_term", "target_term", "pos",
               "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "flight controller"].iloc[0]
assert row["target_term"] == "полётный контроллер", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "noun", "Не извлечён termNote[@type='partOfSpeech']"
assert "telemetry link" in set(glossary_df["source_term"]), \
    "Термин из <tig> без termNote должен быть извлечён"
print("Тест 2 (парсинг TBX) — OK")
glossary_df.head()

Тест 2 (парсинг TBX) — OK


,entry_id,source_term,target_term,pos,definition,subject_field
0,tid-001,flight controller,полётный контроллер,noun,Onboard flight computer.,Demo
1,tid-002,payload,целевая нагрузка,noun,,Demo
2,tid-003,telemetry link,канал телеметрии,,,Demo


In [13]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("demo.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]
assert first["source_segment"].startswith("The flight controller"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("Полётный контроллер"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг TMX) — OK")
tm_df.head()

Тест 3 (парсинг TMX) — OK


,tuid,source_segment,target_segment,domain,src_len,prev_source
0,1,The flight controller stabilises the aircr...,Полётный контроллер стабилизирует аппарат.,Demo,46,
1,2,The operator checks the telemetry link.,Оператор проверяет канал телеметрии.,Demo,39,The flight controller stabilises the aircr...
2,3,The payload is mounted under the fuselage.,Целевая нагрузка устанавливается под фюзел...,Demo,42,The operator checks the telemetry link.


In [14]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------
exact_query = "The operator checks the telemetry link."
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=70.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="The flight controller stabilises the aircraft.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("The engineer checks the telemetry link.",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("telemetry link check",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"
assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"
assert len(calculate_fuzzy_match("telemetry", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"
print("Тест 4 (Fuzzy Match и ICE) — OK")

Тест 4 (Fuzzy Match и ICE) — OK


In [15]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("The flight controller reads the payload status.",
                             glossary_df)
assert "[полётный контроллер]" in marked, "Термин из глоссария не подставлен"
assert "[целевая нагрузка]" in marked, "Подставлен не каждый найденный термин"
assert "flight controller" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Payload mass is 2 kg.", glossary_df)
assert "Payload [целевая нагрузка]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[целевая нагрузка]" not in apply_term_glossary("payloads are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"
print("Тест 5 (подстановка терминов) — OK")
print(marked)

Тест 5 (подстановка терминов) — OK
The flight controller [полётный контроллер] reads the payload [целевая нагрузка] status.


## Блок 6. Работа со своим вариантом

Проверка правильности работы разработанных функций и формирование csv-файлов.

In [16]:
VARIANT = 7
SURNAME = "Klyuchnikova"

TBX_FILE = f"variant_{VARIANT}.tbx"
TMX_FILE = f"variant_{VARIANT}.tmx"

my_glossary = parse_tbx(TBX_FILE)
my_tm = parse_tmx(TMX_FILE)

QUERIES: List[Dict[str, Optional[str]]] = [
    {
        "label": "1. ICE — точное совпадение с совпадением контекста",
        "query": "Field exploration stage includes implementation of geological and seismic exploration operations, drilling of wildcats and then exploratory wells.",
        "context": "Rocks saturated with oil or gas, have lower density than the same rocks containing water.",
    },
    {
        "label": "2. Fuzzy — заменены два слова",
        "query": "Field exploration stage includes implementation of geological and seismic exploration operations, drilling of wildcats and then exploration wells.",
        "context": None,
    },
    {
        "label": "3. No match — сегмент вне предметной области",
        "query": "Wildcats are drilled in order to discover new commercial deposits of oil and gas.",
        "context": None,
    },
]

all_matches: List[Dict[str, Any]] = []
for case in QUERIES:
    matches = calculate_fuzzy_match(case["query"], my_tm, threshold=70.0,
                                    context_before=case["context"])
    print("=" * 100)
    print(case["label"])
    print(f"Запрос: {case['query']}")
    if not matches:
        print("  -> совпадений выше порога 70 % не найдено, сегмент переводится с нуля")
        continue
    for rank, match in enumerate(matches, start=1):
        print(f"  #{rank} {match['score']:6.2f} %  [{match['match_type']:<12}] "
              f"tuid={match['tuid']}")
        print(f"      TM src: {match['source_segment']}")
        print(f"      TM tgt: {match['target_segment']}")
        match["query"] = case["query"]
        all_matches.append(match)

matches_df = pd.DataFrame(all_matches)

GLOSSARY_CSV = "glossary_smartcat_v7.csv"
TM_CSV = "tm_segments_v7.csv"
FUZZY_CSV = "fuzzy_report_v7.csv"

smartcat_df = pd.DataFrame({
    "Source (en)": my_glossary["source_term"],
    "Target (ru)": my_glossary["target_term"],
    "Comment": my_glossary["definition"],
})
smartcat_df.to_csv(GLOSSARY_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)

my_tm[["tuid", "source_segment", "target_segment", "domain"]].to_csv(TM_CSV, index=False, encoding="utf-8-sig")

matches_df[["query", "tuid", "source_segment", "target_segment",
            "score", "lev_ratio", "token_sort_ratio", "match_type"]].to_csv(FUZZY_CSV, index=False, encoding="utf-8-sig")

print("\nЭкспортированные файлы:")
for name in [TBX_PATH, TMX_PATH, GLOSSARY_CSV, TM_CSV, FUZZY_CSV]:
    print(f"  {name:<28} {os.path.getsize(name):>7} байт")

print("=" * 100)
print(f"\nВсего кандидатов выше порога: {len(matches_df)}")
matches_df[["query", "tuid", "score", "lev_ratio", "token_sort_ratio", "match_type"]].head(6)

1. ICE — точное совпадение с совпадением контекста
Запрос: Field exploration stage includes implementation of geological and seismic exploration operations, drilling of wildcats and then exploratory wells.
  #1 100.00 %  [ICE (101%)  ] tuid=4
      TM src: Field exploration stage includes implementation of geological and seismic exploration operations, drilling of wildcats and then exploratory wells.
      TM tgt: Этап разведки месторождения включает проведение геологических и сейсморазведочных работ, бурение поисковых, а затем разведочных скважин.
2. Fuzzy — заменены два слова
Запрос: Field exploration stage includes implementation of geological and seismic exploration operations, drilling of wildcats and then exploration wells.
  #1  97.92 %  [High Fuzzy  ] tuid=4
      TM src: Field exploration stage includes implementation of geological and seismic exploration operations, drilling of wildcats and then exploratory wells.
      TM tgt: Этап разведки месторождения включает проведение 

,query,tuid,score,lev_ratio,token_sort_ratio,match_type
0,Field exploration stage includes implement...,4,100.00,100.00,100.00,ICE (101%)
1,Field exploration stage includes implement...,4,97.92,97.92,97.92,High Fuzzy


## Блок 7. Выводы

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?
   
   В TBX основное внимание уделяется понятиям и их терминологическим эквивалентам, а в TMX — соответствию между фрагментами текста. Поэтому TBX используется для создания глоссариев, а TMX — для повторного использования ранее выполненных переводов.
2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?
   
   Атрибут xml:lang указывает язык термина, но относится к пространству имён XML, поэтому при разборе файла его необходимо извлекать с учётом этого пространства имён. В моей функции для этого используется константа XML_LANG, соответствующая атрибуту xml:lang. При обработке языковой секции значение атрибута приводится к нижнему регистру, а часть после дефиса удаляется.
3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это
   компенсирует Token Sort Ratio?
   
   Levenshtein ratio сравнивает тексты посимвольно, поэтому может давать низкую оценку, если в предложениях переставлены слова. Token Sort Ratio сначала сортирует слова, а затем сравнивает тексты. Благодаря этому он лучше находит похожие предложения, даже если порядок слов отличается.
4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно
   для стоимости проекта в Smartcat?
   
   Обычное совпадение 100 % означает, что предложение полностью совпадает с предложением в памяти переводов. ICE (101 %) означает, что совпадает не только само предложение, но и его контекст — соседние сегменты.
5. Какие проблемы возникли при автоматической подстановке терминов и как вы их
   решили?
   
   При подстановке терминов проблема возникает, когда короткий термин находится внутри более длинного. Чтобы этого избежать, нужно сначала обрабатывать самые длинные термины, а уже потом короткие. Также нужно защищать уже заменённые фрагменты, чтобы повторная подстановка не меняла их и не портила результат.

**Ваши выводы:**

В ходе лабораторной работы я научилась работать с форматами TBX и TMX и использовать память переводов для поиска совпадений. Я изучила методы оценки сходства текстов Levenshtein ratio и Token Sort Ratio и разобралась в различиях между ICE и обычными точными совпадениями. Также я попробовала автоматическую подстановку терминов с помощью глоссария. В результате я лучше поняла, как инструменты автоматизации помогают ускорить перевод и обеспечить единообразие терминологии.
